
## 1. Setup and Configuration
Import the required libraries, locate the EU AI Act PDF, and create folders for saving checkpoints and results.


In [1]:
import os,sys,json,time
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_DIR=Path("/kaggle/input")
PROJECT_DIR=Path("/kaggle/working/LMA_RAG_Project")
CHECKPOINT_DIR=PROJECT_DIR/"checkpoints"
RESULTS_DIR=PROJECT_DIR/"results"
for folder in [PROJECT_DIR,CHECKPOINT_DIR,RESULTS_DIR]:
    folder.mkdir(parents=True,exist_ok=True)

pdf_files=list(INPUT_DIR.rglob("*.pdf"))
print("Python:",sys.version.split()[0])
print("NumPy:",np.__version__,"| Pandas:",pd.__version__)
print("Project folder:",PROJECT_DIR)
print("PDFs found:",len(pdf_files))
for i,pdf in enumerate(pdf_files,1):
    print(f"{i}. {pdf}")

Python: 3.13.15
NumPy: 2.1.3 | Pandas: 2.3.3
Project folder: /kaggle/working/LMA_RAG_Project
PDFs found: 1
1. /kaggle/input/datasets/shaikarish/rag-model-dataset/REGULATION (EU) 20241689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL.pdf



## 2. PDF Text Extraction
Extract the text page by page and save it with page references for later retrieval and verification.


In [2]:
try:
    from pypdf import PdfReader
except ImportError:
    %pip install -q pypdf
    from pypdf import PdfReader

PDF_PATH=pdf_files[0]
reader=PdfReader(str(PDF_PATH))
pages=[]
for page_num,page in enumerate(reader.pages,start=1):
    text=page.extract_text() or ""
    if text.strip():
        pages.append({"page":page_num,"text":text.strip()})

total_chars=sum(len(p["text"]) for p in pages)
page_data=CHECKPOINT_DIR/"pages.json"
with open(page_data,"w",encoding="utf-8") as f:
    json.dump(pages,f,ensure_ascii=False)

print("PDF:",PDF_PATH.name)
print("Total PDF pages:",len(reader.pages))
print("Pages containing text:",len(pages))
print("Total extracted characters:",total_chars)
print("Saved checkpoint:",page_data)
print("\nSample extracted text:\n")
print(pages[0]["text"][:1500] if pages else "No text extracted.")


PDF: REGULATION (EU) 20241689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL.pdf
Total PDF pages: 144
Pages containing text: 144
Total extracted characters: 599631
Saved checkpoint: /kaggle/working/LMA_RAG_Project/checkpoints/pages.json

Sample extracted text:

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of 


## 3. Document Chunking
Split the extracted text into overlapping chunks and preserve page references. Both retrieval pipelines will use this same chunk corpus.


In [3]:
with open(CHECKPOINT_DIR/"pages.json",encoding="utf-8") as f:
    pages=json.load(f)

CHUNK_SIZE=350
CHUNK_OVERLAP=60
words=[]
word_pages=[]
for p in pages:
    page_words=p["text"].split()
    words.extend(page_words)
    word_pages.extend([p["page"]]*len(page_words))

chunks=[]
step=CHUNK_SIZE-CHUNK_OVERLAP
for start in range(0,len(words),step):
    end=min(start+CHUNK_SIZE,len(words))
    if end-start<80:
        break
    chunks.append({"chunk_id":len(chunks),"text":" ".join(words[start:end]),"start_page":word_pages[start],"end_page":word_pages[end-1]})
    if end==len(words):
        break

with open(CHECKPOINT_DIR/"chunks.json","w",encoding="utf-8") as f:
    json.dump(chunks,f,ensure_ascii=False)

print("Total words:",len(words))
print("Chunk size:",CHUNK_SIZE,"words | Overlap:",CHUNK_OVERLAP,"words")
print("Total chunks:",len(chunks))
print("Saved:",CHECKPOINT_DIR/"chunks.json")
print("\nFirst chunk preview:\n",chunks[0]["text"][:1000])
print("\nPage range:",chunks[0]["start_page"],"-",chunks[0]["end_page"])


Total words: 91421
Chunk size: 350 words | Overlap: 60 words
Total chunks: 315
Saved: /kaggle/working/LMA_RAG_Project/checkpoints/chunks.json

First chunk preview:
 REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the European Centr

In [5]:
print("Project exists:",PROJECT_DIR.exists())
print("Checkpoint folder exists:",CHECKPOINT_DIR.exists())
print("Checkpoint path:",CHECKPOINT_DIR)
print("Project contents:",[p.name for p in PROJECT_DIR.iterdir()])

Project exists: True
Checkpoint folder exists: True
Checkpoint path: /kaggle/working/LMA_RAG_Project/checkpoints
Project contents: ['results', 'checkpoints']



## 4. Embedding Generation
Convert the document chunks into dense vectors using a pretrained Bi-Encoder. Save the embeddings so they can be reused after a runtime restart.


In [7]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    %pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

with open(CHECKPOINT_DIR/"chunks.json",encoding="utf-8") as f:
    chunks=json.load(f)

EMBEDDING_MODEL="sentence-transformers/all-MiniLM-L6-v2"
model=SentenceTransformer(EMBEDDING_MODEL,device="cpu")
texts=[chunk["text"] for chunk in chunks]
start_time=time.perf_counter()
embeddings=model.encode(texts,batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
embedding_time=time.perf_counter()-start_time

np.save(CHECKPOINT_DIR/"embeddings.npy",embeddings)
with open(CHECKPOINT_DIR/"embedding_config.json","w",encoding="utf-8") as f:
    json.dump({"model":EMBEDDING_MODEL,"chunk_count":len(chunks),"dimension":int(embeddings.shape[1]),"normalized":True},f,indent=2)

print("Model:",EMBEDDING_MODEL)
print("Chunks embedded:",len(embeddings))
print("Embedding shape:",embeddings.shape)
print("Embedding time:",round(embedding_time,2),"seconds")
print("Saved embeddings:",CHECKPOINT_DIR/"embeddings.npy")
print("Saved config:",CHECKPOINT_DIR/"embedding_config.json")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Chunks embedded: 315
Embedding shape: (315, 384)
Embedding time: 14.56 seconds
Saved embeddings: /kaggle/working/LMA_RAG_Project/checkpoints/embeddings.npy
Saved config: /kaggle/working/LMA_RAG_Project/checkpoints/embedding_config.json



## 5. Shared Vector Retrieval
Load the saved chunks and embeddings, then retrieve the Top 10 chunks using cosine similarity. This same retrieval function will supply candidates to both pipelines.


In [9]:
with open(CHECKPOINT_DIR/"chunks.json",encoding="utf-8") as f:
    chunks=json.load(f)
embeddings=np.load(CHECKPOINT_DIR/"embeddings.npy")
assert len(chunks)==embeddings.shape[0],"Chunk and embedding counts do not match."

def retrieve(query,top_k=10):
    query_vector=model.encode([query],normalize_embeddings=True,convert_to_numpy=True)[0]
    scores=embeddings@query_vector
    ranked=np.argsort(scores)[::-1][:top_k]
    return [{"rank":rank+1,"chunk_id":int(i),"score":float(scores[i]),"text":chunks[i]["text"],"start_page":chunks[i]["start_page"],"end_page":chunks[i]["end_page"]} for rank,i in enumerate(ranked)]

sample_query="What types of artificial intelligence practices are prohibited under the AI Act?"
start_time=time.perf_counter()
sample_results=retrieve(sample_query,top_k=10)
retrieval_time=time.perf_counter()-start_time

print("Query:",sample_query)
print("Retrieved:",len(sample_results),"chunks")
print("Retrieval time:",round(retrieval_time*1000,2),"ms")
for result in sample_results[:3]:
    print(f"\nRank {result['rank']} | Chunk {result['chunk_id']} | Similarity {result['score']:.4f} | Pages {result['start_page']}-{result['end_page']}")
    print(result["text"][:350].replace("\n"," "))


Query: What types of artificial intelligence practices are prohibited under the AI Act?
Retrieved: 10 chunks
Retrieval time: 45.18 ms

Rank 1 | Chunk 128 | Similarity 0.6404 | Pages 44-45
of application of this Regulation. Therefore the provisions on penalties should apply from 2 August 2025. (180) The European Data Protection Supervisor and the European Data Protection Board were consulted in accordance with Article 42(1) and (2) of Regulation (EU) 2018/1725 and delivered their joint opinion on 18 June 2021, HAVE ADOPTED THIS REGUL

Rank 2 | Chunk 40 | Similarity 0.6233 | Pages 14-14
taking into account both the severity of the possible harm and its probability of occurrence and they are used in a number of specifically pre-defined areas specified in this Regulation. The identification of those systems is based on the same methodology and criteria envisaged also for any future amendments of the list of high-risk AI systems that

Rank 3 | Chunk 285 | Similarity 0.6155 | Pages 121-122
R

In [10]:
for r in sample_results:
    print(f"\nRank {r['rank']} | Chunk {r['chunk_id']} | Score {r['score']:.4f} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:600].replace("\n"," "))



Rank 1 | Chunk 128 | Score 0.6404 | Pages 44-45
of application of this Regulation. Therefore the provisions on penalties should apply from 2 August 2025. (180) The European Data Protection Supervisor and the European Data Protection Board were consulted in accordance with Article 42(1) and (2) of Regulation (EU) 2018/1725 and delivered their joint opinion on 18 June 2021, HAVE ADOPTED THIS REGULATION: CHAPTER I GENERAL PROVISIONS Article 1 Subject matter` 1. The purpose of this Regulation is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence (AI), while ensuring 

Rank 2 | Chunk 40 | Score 0.6233 | Pages 14-14
taking into account both the severity of the possible harm and its probability of occurrence and they are used in a number of specifically pre-defined areas specified in this Regulation. The identification of those systems is based on the same methodology and criteria envisaged also for any future ame


## 6. Cross-Encoder Re-Ranking
Use a Cross-Encoder to score the query against each of the ten retrieved chunks and reorder them by relevance.


In [11]:
try:
    from sentence_transformers import CrossEncoder
except ImportError:
    %pip install -q sentence-transformers
    from sentence_transformers import CrossEncoder

CROSS_ENCODER_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder=CrossEncoder(CROSS_ENCODER_NAME,device="cpu")
print("Cross-Encoder loaded:",CROSS_ENCODER_NAME)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [12]:
# Score the same Top 10 candidates and sort by relevance
pairs=[(sample_query,r["text"]) for r in sample_results]

start_time=time.perf_counter()
ce_scores=cross_encoder.predict(pairs,show_progress_bar=False)
rerank_time=time.perf_counter()-start_time

reranked=sorted(
    [{**r,"ce_score":float(score)} for r,score in zip(sample_results,ce_scores)],
    key=lambda x:x["ce_score"],reverse=True
)

print("Cross-Encoder scoring time:",round(rerank_time*1000,2),"ms")
print("\nRe-ranked results:")
for rank,r in enumerate(reranked,1):
    r["new_rank"]=rank
    print(f"Rank {rank} | Chunk {r['chunk_id']} | CE score {r['ce_score']:.4f} | Original rank {r['rank']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:250].replace("\n"," "),"\n")

Cross-Encoder scoring time: 1040.34 ms

Re-ranked results:
Rank 1 | Chunk 24 | CE score 3.2809 | Original rank 8 | Pages 9-9
for such AI practices are complementary to the provisions contained in Directive 2005/29/EC of the European Parliament and of the Council (17), in particular unfair commercial practices leading to economic or financial harms to consumers are prohibit 

Rank 2 | Chunk 22 | CE score 1.7285 | Original rank 4 | Pages 8-8
including industry, academia, civil society and standardisation organisations, are encouraged to take into account, as appropriate, the ethical principles for the development of voluntary best practices and standards. (28) Aside from the many benefic 

Rank 3 | Chunk 284 | CE score 0.2074 | Original rank 7 | Pages 121-121
‘3. When adopting the implementing acts pursuant to paragraph 2, concerning artificial intelligence systems which are safety components within the meaning of Regulation (EU) 2024/1689 of the European Parliament and of the Council (*

## 7. Baseline vs. Re-Ranked Results

Compare the Top 3 chunks selected by the Bi-Encoder with the Top 3 selected after Cross-Encoder re-ranking. Check whether re-ranking places more relevant information higher in the results.

In [13]:
# Compare the Top 3 selected by each pipeline
baseline_top3=sample_results[:3]
reranked_top3=reranked[:3]

print("BASELINE: Bi-Encoder Top 3")
for r in baseline_top3:
    print(f"Chunk {r['chunk_id']} | Original rank {r['rank']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:400].replace("\n"," "),"\n")

print("\nCROSS-ENCODER: Re-ranked Top 3")
for r in reranked_top3:
    print(f"Chunk {r['chunk_id']} | Original rank {r['rank']} -> New rank {r['new_rank']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:400].replace("\n"," "),"\n")

BASELINE: Bi-Encoder Top 3
Chunk 128 | Original rank 1 | Pages 44-45
of application of this Regulation. Therefore the provisions on penalties should apply from 2 August 2025. (180) The European Data Protection Supervisor and the European Data Protection Board were consulted in accordance with Article 42(1) and (2) of Regulation (EU) 2018/1725 and delivered their joint opinion on 18 June 2021, HAVE ADOPTED THIS REGULATION: CHAPTER I GENERAL PROVISIONS Article 1 Subj 

Chunk 40 | Original rank 2 | Pages 14-14
taking into account both the severity of the possible harm and its probability of occurrence and they are used in a number of specifically pre-defined areas specified in this Regulation. The identification of those systems is based on the same methodology and criteria envisaged also for any future amendments of the list of high-risk AI systems that the Commission should be empowered to adopt, via  

Chunk 285 | Original rank 3 | Pages 121-122
Regulation shall be taken into account i

In [14]:
# Save the original and re-ranked results for the report
comparison=[]
for r in sample_results:
    match=next(x for x in reranked if x["chunk_id"]==r["chunk_id"])
    comparison.append({
        "chunk_id":r["chunk_id"],
        "original_rank":r["rank"],
        "cross_encoder_rank":match["new_rank"],
        "bi_encoder_score":r["score"],
        "cross_encoder_score":match["ce_score"],
        "start_page":r["start_page"],
        "end_page":r["end_page"]
    })

comparison_df=pd.DataFrame(comparison).sort_values("cross_encoder_rank")
comparison_df.to_csv(RESULTS_DIR/"ranking_comparison.csv",index=False)
display(comparison_df)
print("Saved:",RESULTS_DIR/"ranking_comparison.csv")

,chunk_id,original_rank,cross_encoder_rank,bi_encoder_score,cross_encoder_score,start_page,end_page
7,24,8,1,0.595963,3.280937,9,9
3,22,4,2,0.614457,1.728470,8,8
6,284,7,3,0.606622,0.207374,121,121
0,128,1,4,0.640393,-0.654623,44,45
2,285,3,5,0.615521,-1.095464,121,122
5,20,6,6,0.608109,-1.980155,7,8
9,208,10,7,0.591424,-2.606877,82,83
4,120,5,8,0.610291,-4.116801,41,42
1,40,2,9,0.623288,-4.896280,14,14
8,3,9,10,0.592138,-5.127725,2,2


Saved: /kaggle/working/LMA_RAG_Project/results/ranking_comparison.csv


## 8. Configure the Gemini API
Load the API key securely from Kaggle Secrets and initialize the Gemini client.

In [15]:
!pip install -q google-genai

In [16]:
from kaggle_secrets import UserSecretsClient
from google import genai

api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


### Shared prompt and generation helpers
Define the common prompt, context formatter, and Gemini generation function once. Both pipelines use the same prompt structure and model.

In [ ]:
from google.genai import types, errors
import time

prompt_template = """Answer the question using only the supplied context.
If the context does not contain enough information, say so.
Do not invent facts. Keep the answer concise.

Context:
{context}

Question: {query}

Answer:"""

def build_context(results):
    return "\n\n".join(
        f"[Chunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}]\n{r['text']}"
        for r in results
    )

def generate_answer(prompt, model_name="gemini-3.5-flash-lite"):
    start = time.perf_counter()
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=500,
            http_options=types.HttpOptions(timeout=60000)
        )
    )
    return response.text, time.perf_counter() - start

# Prepare the baseline prompt before the first answer-generation cell.
baseline_top3 = sample_results[:3]
baseline_prompt = prompt_template.format(
    context=build_context(baseline_top3),
    query=sample_query
)
print("Prompt and answer-generation helpers are ready.")


In [23]:
start_time = time.perf_counter()

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="In one sentence, define RAG.",
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=100,
        http_options=types.HttpOptions(timeout=60000)
    )
)

print(response.text)
print("Test response time:", round(time.perf_counter() - start_time, 2), "seconds")

Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves the accuracy and reliability of generative AI models by fetching relevant information from an external knowledge base before generating a response.
Test response time: 1.5 seconds


## 9. Baseline Answer Generation
Use the original Top 3 Bi-Encoder chunks as context and generate Answer 1 with Gemini. Record the response and generation time.

In [24]:
baseline_answer, baseline_generation_time = generate_answer(
    baseline_prompt,
    model_name="gemini-3.5-flash-lite"
)

print("ANSWER 1 — BASELINE")
print(baseline_answer)
print("\nGeneration time:", round(baseline_generation_time, 2), "seconds")

with open(RESULTS_DIR / "baseline_answer.json", "w", encoding="utf-8") as f:
    json.dump({
        "query": sample_query,
        "model": "gemini-3.5-flash-lite",
        "answer": baseline_answer,
        "generation_time_seconds": baseline_generation_time,
        "chunks": [r["chunk_id"] for r in baseline_top3]
    }, f, indent=2, ensure_ascii=False)

print("Baseline answer saved successfully.")


ANSWER 1 — BASELINE
Based on the supplied context, the text mentions "prohibitions of certain AI practices" (Article 1, paragraph 2(b)) and the "list of prohibited AI practices laid down in Article 5" (Article 112, paragraph 1), but the context does not contain the specific details or types of AI practices that are prohibited.

Generation time: 1.34 seconds
Baseline answer saved successfully.


## 10. Re-Ranked Answer Generation
Use the Top 3 Cross-Encoder re-ranked chunks as context and generate Answer 2 with the same Gemini model and prompt structure.

In [25]:
reranked_prompt = prompt_template.format(
    context=build_context(reranked_top3),
    query=sample_query
)

reranked_answer, reranked_generation_time = generate_answer(
    reranked_prompt,
    model_name="gemini-3.5-flash-lite"
)

print("ANSWER 2 — RE-RANKED")
print(reranked_answer)
print("\nGeneration time:", round(reranked_generation_time, 2), "seconds")

with open(RESULTS_DIR / "reranked_answer.json", "w", encoding="utf-8") as f:
    json.dump({
        "query": sample_query,
        "model": "gemini-3.5-flash-lite",
        "answer": reranked_answer,
        "generation_time_seconds": reranked_generation_time,
        "chunks": [r["chunk_id"] for r in reranked_top3]
    }, f, indent=2, ensure_ascii=False)

print("Re-ranked answer saved successfully.")


ANSWER 2 — RE-RANKED
Based on the provided context, the prohibited artificial intelligence practices include:

* **Manipulative, exploitative, and social control practices** that contradict Union values (such as human dignity, freedom, equality, democracy, the rule of law, and fundamental rights). This specifically includes AI systems with the objective or effect of materially distorting human behaviour in a manner that causes significant harms (such as adverse impacts on physical, psychological health, or financial interests) by deploying subliminal components beyond human perception or other deceptive techniques that subvert or impair autonomy, decision-making, or free choice. It also includes AI systems that exploit the vulnerabilities of persons or specific groups due to their age or disability.
* **Biometric categorisation systems** based on natural persons' biometric data (such as faces or fingerprints) used to deduce or infer an individual's political opinions, trade union membe

## 11. Answer Comparison
Compare both generated answers, their generation times, and the retrieved chunks used by each pipeline.

In [26]:
answer_comparison = pd.DataFrame([
    {
        "pipeline": "Baseline",
        "chunks": [r["chunk_id"] for r in baseline_top3],
        "generation_time_seconds": round(baseline_generation_time, 2),
        "answer": baseline_answer
    },
    {
        "pipeline": "Cross-Encoder Re-ranked",
        "chunks": [r["chunk_id"] for r in reranked_top3],
        "generation_time_seconds": round(reranked_generation_time, 2),
        "answer": reranked_answer
    }
])

answer_comparison.to_csv(RESULTS_DIR / "answer_comparison.csv", index=False)
display(answer_comparison[["pipeline", "chunks", "generation_time_seconds"]])
print("Saved:", RESULTS_DIR / "answer_comparison.csv")


,pipeline,chunks,generation_time_seconds
0,Baseline,"[128, 40, 285]",1.34
1,Cross-Encoder Re-ranked,"[24, 22, 284]",2.21


Saved: /kaggle/working/LMA_RAG_Project/results/answer_comparison.csv


In [27]:
# Inspect the complete context supplied to Answer 2
for r in reranked_top3:
    print(f"\n{'='*70}")
    print(f"Chunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}")
    print("="*70)
    print(r["text"])


Chunk 24 | Pages 9-9
for such AI practices are complementary to the provisions contained in Directive 2005/29/EC of the European Parliament and of the Council (17), in particular unfair commercial practices leading to economic or financial harms to consumers are prohibited under all circumstances, irrespective of whether they are put in place through AI systems or otherwise. The prohibitions of manipulative and exploitative practices in this Regulation should not affect lawful practices in the context of medical treatment such as psychological treatment of a mental disease or physical rehabilitation, when those practices are carried out in accordance with the applicable law and medical standards, for example explicit consent of the individuals or their legal representatives. In addition, common and legitimate commercial practices, for example in the field of advertising, that comply with the applicable law should not, in themselves, be regarded as constituting harmful manipulative AI-

## 12. Second Failure Query: Exceptions and Restrictions
Test whether Cross-Encoder re-ranking can promote a specific exception or restriction that the baseline retrieval misses.

In [29]:
second_query = (
    "What exceptions or limitations apply to the prohibition on "
    "biometric categorisation based on sensitive personal characteristics?"
)

second_results = retrieve(second_query, top_k=10)

second_scores = cross_encoder.predict([
    (second_query, r["text"]) for r in second_results
])

second_reranked = sorted(
    [{**r, "ce_score": float(score)} for r, score in zip(second_results, second_scores)],
    key=lambda x: x["ce_score"],
    reverse=True
)

print("QUERY:", second_query)

print("\nBASELINE TOP 3")
for r in second_results[:3]:
    print(f"\nChunk {r['chunk_id']} | Rank {r['rank']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:350].replace("\n", " "))

print("\nRE-RANKED TOP 3")
for rank, r in enumerate(second_reranked[:3], 1):
    print(f"\nChunk {r['chunk_id']} | Original rank {r['rank']} -> New rank {rank} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:350].replace("\n", " "))

QUERY: What exceptions or limitations apply to the prohibition on biometric categorisation based on sensitive personal characteristics?

BASELINE TOP 3

Chunk 11 | Rank 1 | Pages 4-5
EN OJ L, 12.7.2024 4/144 ELI: http://data.europa.eu/eli/reg/2024/1689/oj (15) Regulation (EU) 2022/2065 of the European Parliament and of the Council of 19 October 2022 on a Single Market For Digital Services and amending Directive 2000/31/EC (Digital Services Act) (OJ L 277, 27.10.2022, p. 1). (16) The notion of ‘biometric categorisation’ referred

Chunk 32 | Rank 2 | Pages 11-12
the use of remote biometric identification for purposes other than law enforcement has already been subject to prohibition decisions by national data protection authorities. (40) In accordance with Article 6a of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the area of freedom, security and justice, as annexed to th

Chunk 24 | Rank 3 | Pages 9-9
for such AI practices are complementary to the prov

In [30]:
third_query = (
    "When is biometric categorisation of biometric datasets allowed, "
    "such as labelling or filtering images by hair colour or eye colour?"
)

third_results = retrieve(third_query, top_k=10)
third_scores = cross_encoder.predict([
    (third_query, r["text"]) for r in third_results
])

third_reranked = sorted(
    [{**r, "ce_score": float(score)} for r, score in zip(third_results, third_scores)],
    key=lambda x: x["ce_score"],
    reverse=True
)

for title, results in [
    ("BASELINE TOP 3", third_results[:3]),
    ("RE-RANKED TOP 3", third_reranked[:3])
]:
    print(f"\n{title}")
    for rank, r in enumerate(results, 1):
        print(
            f"\nChunk {r['chunk_id']} | Rank {rank} | "
            f"Original rank {r['rank']} | Pages "
            f"{r['start_page']}-{r['end_page']}"
        )
        print(r["text"][:450].replace("\n", " "))


BASELINE TOP 3

Chunk 11 | Rank 1 | Original rank 1 | Pages 4-5
EN OJ L, 12.7.2024 4/144 ELI: http://data.europa.eu/eli/reg/2024/1689/oj (15) Regulation (EU) 2022/2065 of the European Parliament and of the Council of 19 October 2022 on a Single Market For Digital Services and amending Directive 2000/31/EC (Digital Services Act) (OJ L 277, 27.10.2022, p. 1). (16) The notion of ‘biometric categorisation’ referred to in this Regulation should be defined as assigning natural persons to specific categories on the 

Chunk 32 | Rank 2 | Original rank 2 | Pages 11-12
the use of remote biometric identification for purposes other than law enforcement has already been subject to prohibition decisions by national data protection authorities. (40) In accordance with Article 6a of Protocol No 21 on the position of the United Kingdom and Ireland in respect of the area of freedom, security and justice, as annexed to the TEU and to the TFEU, Ireland is not bound by the rules laid down in Article 5(1),

In [31]:
for r in third_results:
    if r["chunk_id"] == 11:
        print(r["text"])
        break

EN OJ L, 12.7.2024 4/144 ELI: http://data.europa.eu/eli/reg/2024/1689/oj (15) Regulation (EU) 2022/2065 of the European Parliament and of the Council of 19 October 2022 on a Single Market For Digital Services and amending Directive 2000/31/EC (Digital Services Act) (OJ L 277, 27.10.2022, p. 1). (16) The notion of ‘biometric categorisation’ referred to in this Regulation should be defined as assigning natural persons to specific categories on the basis of their biometric data. Such specific categories can relate to aspects such as sex, age, hair colour, eye colour, tattoos, behavioural or personality traits, language, religion, membership of a national minority, sexual or political orientation. This does not include biometric categorisation systems that are a purely ancillary feature intrinsically linked to another commercial service, meaning that the feature cannot, for objective technical reasons, be used without the principal service, and the integration of that feature or functional

In [32]:
for new_rank, r in enumerate(third_reranked[:3], start=1):
    print(f"\n{'='*60}")
    print(f"New Rank: {new_rank}")
    print(f"Chunk ID: {r['chunk_id']}")
    print(f"Original Bi-Encoder Rank: {r['rank']}")
    print(f"Cross-Encoder Score: {r['ce_score']:.4f}")
    print(f"Pages: {r['start_page']}-{r['end_page']}")
    print(r["text"])


New Rank: 1
Chunk ID: 11
Original Bi-Encoder Rank: 1
Cross-Encoder Score: 2.5680
Pages: 4-5
EN OJ L, 12.7.2024 4/144 ELI: http://data.europa.eu/eli/reg/2024/1689/oj (15) Regulation (EU) 2022/2065 of the European Parliament and of the Council of 19 October 2022 on a Single Market For Digital Services and amending Directive 2000/31/EC (Digital Services Act) (OJ L 277, 27.10.2022, p. 1). (16) The notion of ‘biometric categorisation’ referred to in this Regulation should be defined as assigning natural persons to specific categories on the basis of their biometric data. Such specific categories can relate to aspects such as sex, age, hair colour, eye colour, tattoos, behavioural or personality traits, language, religion, membership of a national minority, sexual or political orientation. This does not include biometric categorisation systems that are a purely ancillary feature intrinsically linked to another commercial service, meaning that the feature cannot, for objective technical reas

In [33]:
# Generate answers for the third query using both pipelines

third_baseline_top3 = third_results[:3]
third_reranked_top3 = third_reranked[:3]

third_baseline_prompt = prompt_template.format(
    context=build_context(third_baseline_top3),
    query=third_query
)

third_reranked_prompt = prompt_template.format(
    context=build_context(third_reranked_top3),
    query=third_query
)

# Use the same model for a fair comparison
third_baseline_answer, third_baseline_time = generate_answer(
    third_baseline_prompt,
    model_name="gemini-3.5-flash-lite"
)

third_reranked_answer, third_reranked_time = generate_answer(
    third_reranked_prompt,
    model_name="gemini-3.5-flash-lite"
)

print("BASELINE ANSWER (Top 3 Bi-Encoder)")
print(third_baseline_answer)
print(f"Generation time: {third_baseline_time:.2f} seconds")

print("\n" + "=" * 60)

print("RE-RANKED ANSWER (Top 3 Cross-Encoder)")
print(third_reranked_answer)
print(f"Generation time: {third_reranked_time:.2f} seconds")

BASELINE ANSWER (Top 3 Bi-Encoder)
Based on the supplied context, the definition states that biometric categorisation assigns natural persons to specific categories (such as hair colour or eye colour) on the basis of their biometric data, and does not include systems that are a purely ancillary feature intrinsically linked to another commercial service (meaning the feature cannot, for objective technical reasons, be used without the principal service, and its integration is not a means to circumvent the rules). Examples include filters categorizing facial or body features used on online marketplaces to preview products, or filters on social network services used to add or modify pictures or videos. 

However, the provided context does not explicitly specify general rules on when biometric categorisation of biometric datasets is broadly "allowed."
Generation time: 1.91 seconds

RE-RANKED ANSWER (Top 3 Cross-Encoder)
Based on the provided context, the labelling, filtering, or categorisat

In [34]:
# Save the third query's answer comparison

third_answer_comparison = pd.DataFrame([
    {
        "pipeline": "Baseline",
        "query": third_query,
        "chunks": [r["chunk_id"] for r in third_baseline_top3],
        "generation_time_seconds": round(third_baseline_time, 2),
        "answer": third_baseline_answer
    },
    {
        "pipeline": "Cross-Encoder Re-ranked",
        "query": third_query,
        "chunks": [r["chunk_id"] for r in third_reranked_top3],
        "generation_time_seconds": round(third_reranked_time, 2),
        "answer": third_reranked_answer
    }
])

third_answer_comparison.to_csv(
    RESULTS_DIR / "third_answer_comparison.csv",
    index=False
)

third_ranking_comparison = pd.DataFrame([
    {
        "chunk_id": r["chunk_id"],
        "original_rank": r["rank"],
        "cross_encoder_rank": new_rank,
        "cross_encoder_score": r["ce_score"],
        "start_page": r["start_page"],
        "end_page": r["end_page"]
    }
    for new_rank, r in enumerate(third_reranked, start=1)
])

third_ranking_comparison.to_csv(
    RESULTS_DIR / "third_ranking_comparison.csv",
    index=False
)

print("Third-query results saved successfully.")

Third-query results saved successfully.


In [ ]:
# Check the timing variables already recorded

print("Baseline generation time:", baseline_generation_time)
print("Re-ranked generation time:", reranked_generation_time)
print("Cross-Encoder scoring time:", ce_time if "ce_time" in globals() else "Variable not found")

In [35]:
# Check the timing variables already recorded

print("Baseline generation time:", baseline_generation_time)
print("Re-ranked generation time:", reranked_generation_time)
print("Cross-Encoder scoring time:", ce_time if "ce_time" in globals() else "Variable not found")

Baseline generation time: 1.3407798170001115
Re-ranked generation time: 2.212718943999789
Cross-Encoder scoring time: Variable not found


In [36]:
import time

# Use the same query for both pipelines
query = sample_query

# 1. Baseline: retrieve Top 10 and select Top 3
start = time.perf_counter()
baseline_candidates = retrieve(query, top_k=10)
baseline_top3_timed = baseline_candidates[:3]
baseline_retrieval_time = time.perf_counter() - start

# 2. Re-ranking: retrieve Top 10, score all 10, select new Top 3
start = time.perf_counter()
rerank_candidates = retrieve(query, top_k=10)

pairs = [(query, r["text"]) for r in rerank_candidates]
cross_scores = cross_encoder.predict(pairs)

rerank_scored = [
    {**r, "ce_score": float(score)}
    for r, score in zip(rerank_candidates, cross_scores)
]
rerank_scored.sort(key=lambda x: x["ce_score"], reverse=True)
rerank_top3_timed = rerank_scored[:3]

reranking_pipeline_time = time.perf_counter() - start

print(f"Baseline retrieval time: {baseline_retrieval_time:.4f} seconds")
print(f"Re-ranked retrieval + Cross-Encoder time: {reranking_pipeline_time:.4f} seconds")
print(f"Cross-Encoder scored {len(pairs)} candidate chunks.")

Baseline retrieval time: 0.0209 seconds
Re-ranked retrieval + Cross-Encoder time: 1.0390 seconds
Cross-Encoder scored 10 candidate chunks.


In [39]:
import time
import pandas as pd

query = sample_query

# Measure Bi-Encoder retrieval for the re-ranked pipeline
start = time.perf_counter()
rerank_candidates = retrieve(query, top_k=10)
reranked_retrieval_time = time.perf_counter() - start

# Measure Cross-Encoder scoring and sorting only
pairs = [(query, r["text"]) for r in rerank_candidates]

start = time.perf_counter()
cross_scores = cross_encoder.predict(pairs)

rerank_scored = [
    {**r, "ce_score": float(score)}
    for r, score in zip(rerank_candidates, cross_scores)
]
rerank_scored.sort(key=lambda x: x["ce_score"], reverse=True)
rerank_top3_timed = rerank_scored[:3]

cross_encoder_time = time.perf_counter() - start

# Build a correctly labelled latency table
latency_df = pd.DataFrame([
    {
        "pipeline": "Baseline",
        "retrieval_s": baseline_retrieval_time,
        "cross_encoder_s": 0.0,
        "generation_s": baseline_generation_time,
        "total_s": baseline_retrieval_time + baseline_generation_time
    },
    {
        "pipeline": "Cross-Encoder Re-ranked",
        "retrieval_s": reranked_retrieval_time,
        "cross_encoder_s": cross_encoder_time,
        "generation_s": reranked_generation_time,
        "total_s": (
            reranked_retrieval_time
            + cross_encoder_time
            + reranked_generation_time
        )
    }
])

latency_df.to_csv(RESULTS_DIR / "latency_comparison.csv", index=False)
display(latency_df.round(4))

print("Updated latency_comparison.csv")

,pipeline,retrieval_s,cross_encoder_s,generation_s,total_s
0,Baseline,0.0209,0.0000,1.3408,1.3617
1,Cross-Encoder Re-ranked,0.0221,1.0499,2.2127,3.2847


Updated latency_comparison.csv


### Step 14: Keyword Distractor Test

In this experiment, we test whether the Bi-Encoder retrieves passages that contain similar keywords but do not directly answer the query. We compare the original Top 3 chunks with the Cross-Encoder re-ranked Top 3 to check whether re-ranking improves relevance.

In [40]:
# Test whether re-ranking improves retrieval for a question about AI Act penalties
distractor_query = "What administrative fines can be imposed on companies for violating the AI Act, and how are the fine amounts determined?"
distractor_results = retrieve(distractor_query, top_k=10)
distractor_scores = cross_encoder.predict([(distractor_query, r["text"]) for r in distractor_results])
distractor_reranked = sorted([{**r, "ce_score": float(s)} for r, s in zip(distractor_results, distractor_scores)], key=lambda x: x["ce_score"], reverse=True)
for new_rank, r in enumerate(distractor_reranked, 1):
    print(f"\nNew rank: {new_rank} | Chunk: {r['chunk_id']} | Original rank: {r['rank']} | CE score: {r['ce_score']:.4f} | Pages: {r['start_page']}-{r['end_page']}")
    print(r["text"][:700])


New rank: 1 | Chunk: 275 | Original rank: 1 | CE score: 3.9723 | Pages: 116-116
its total worldwide annual turnover for the preceding financial year, whichever is higher. 6. In the case of SMEs, including start-ups, each fine referred to in this Article shall be up to the percentages or amount referred to in paragraphs 3, 4 and 5, whichever thereof is lower. 7. When deciding whether to impose an administrative fine and when deciding on the amount of the administrative fine in each individual case, all relevant circumstances of the specific situation shall be taken into account and, as appropriate, regard shall be given to the following: (a) the nature, gravity and duration of the infringement and of its consequences, taking into account the purpose of the AI system, as 

New rank: 2 | Chunk: 122 | Original rank: 2 | CE score: 3.9430 | Pages: 42-42
penalties for their infringement, and to respect the ne bis in idem principle. In order to strengthen and harmonise administrative penaltie

### Inspect the Baseline Results

We inspect the original Top 3 chunks to determine whether they answer the query adequately before deciding if this is a genuine keyword-distractor failure.

In [41]:
# Inspect the original Top 3 chunks
for r in distractor_results[:3]:
    print(f"\nChunk {r['chunk_id']} | Original rank: {r['rank']} | Pages: {r['start_page']}-{r['end_page']}")
    print(r["text"])


Chunk 275 | Original rank: 1 | Pages: 116-116
its total worldwide annual turnover for the preceding financial year, whichever is higher. 6. In the case of SMEs, including start-ups, each fine referred to in this Article shall be up to the percentages or amount referred to in paragraphs 3, 4 and 5, whichever thereof is lower. 7. When deciding whether to impose an administrative fine and when deciding on the amount of the administrative fine in each individual case, all relevant circumstances of the specific situation shall be taken into account and, as appropriate, regard shall be given to the following: (a) the nature, gravity and duration of the infringement and of its consequences, taking into account the purpose of the AI system, as well as, where appropriate, the number of affected persons and the level of damage suffered by them; (b) whether administrative fines have already been applied by other market surveillance authorities to the same operator for the same infringement; (c) 

### Negation and Constraint Test

This experiment tests whether the retrieval system distinguishes prohibited AI practices from specific exceptions or conditions under which a practice is not covered by a prohibition. We compare the original Top 3 with the Cross-Encoder re-ranked Top 3.

In [42]:
# Test retrieval of a specific exception to a prohibition
negation_query = "Which biometric categorisation activities are not prohibited under the AI Act, and what conditions apply?"
negation_results = retrieve(negation_query, top_k=10)
negation_scores = cross_encoder.predict([(negation_query, r["text"]) for r in negation_results])
negation_reranked = sorted([{**r, "ce_score": float(s)} for r, s in zip(negation_results, negation_scores)], key=lambda x: x["ce_score"], reverse=True)
for new_rank, r in enumerate(negation_reranked, 1):
    print(f"\nNew rank: {new_rank} | Chunk: {r['chunk_id']} | Original rank: {r['rank']} | CE score: {r['ce_score']:.4f} | Pages: {r['start_page']}-{r['end_page']}")
    print(r["text"][:600])


New rank: 1 | Chunk: 24 | Original rank: 2 | CE score: 4.8172 | Pages: 9-9
for such AI practices are complementary to the provisions contained in Directive 2005/29/EC of the European Parliament and of the Council (17), in particular unfair commercial practices leading to economic or financial harms to consumers are prohibited under all circumstances, irrespective of whether they are put in place through AI systems or otherwise. The prohibitions of manipulative and exploitative practices in this Regulation should not affect lawful practices in the context of medical treatment such as psychological treatment of a mental disease or physical rehabilitation, when those pr

New rank: 2 | Chunk: 44 | Original rank: 4 | CE score: 4.0532 | Pages: 15-16
AI systems intended to be used for biometric verification, including authentication, the sole purpose of which is to confirm that a specific natural person is who that person claims to be and to confirm the identity of a natural person for the s

### Compare Answers for the Negation Query

We compare the answers generated from the original Top 3 and the Cross-Encoder re-ranked Top 3 to determine whether re-ranking improves the identification of exceptions and conditions.

In [43]:
# Generate answers using the same prompt and LLM
negation_baseline = negation_results[:3]
negation_top3 = negation_reranked[:3]
negation_baseline_prompt = prompt_template.format(context=build_context(negation_baseline), query=negation_query)
negation_reranked_prompt = prompt_template.format(context=build_context(negation_top3), query=negation_query)
negation_baseline_answer, negation_baseline_time = generate_answer(negation_baseline_prompt, model_name="gemini-3.5-flash-lite")
negation_reranked_answer, negation_reranked_time = generate_answer(negation_reranked_prompt, model_name="gemini-3.5-flash-lite")
print("BASELINE ANSWER\n", negation_baseline_answer)
print(f"Generation time: {negation_baseline_time:.2f} seconds")
print("\nRE-RANKED ANSWER\n", negation_reranked_answer)
print(f"Generation time: {negation_reranked_time:.2f} seconds")

BASELINE ANSWER
 Based on the supplied context, the prohibition on biometric categorisation systems (specifically those used to deduce or infer political opinions, trade union membership, religious or philosophical beliefs, race, sex life, or sexual orientation) does not cover the lawful labelling, filtering, or categorisation of biometric data sets acquired in line with Union or national law according to biometric data, such as the sorting of images according to hair colour or eye colour, which can, for example, be used in the area of law enforcement. 

Additionally, biometric categorisation systems that are a purely ancillary feature intrinsically linked to another commercial service are not included in the definition of biometric categorisation, provided they meet the following conditions:
* The feature cannot, for objective technical reasons, be used without the principal service.
* The integration of the feature or functionality is not a means to circumvent the applicability of th

In [44]:
keyword_query = "Can an AI system be used to infer a person's emotions in the workplace, and what exceptions apply?"

keyword_results = retrieve(keyword_query, top_k=10)
keyword_scores = cross_encoder.predict([
    (keyword_query, r["text"]) for r in keyword_results
])

keyword_reranked = sorted(
    [{**r, "ce_score": float(s)}
     for r, s in zip(keyword_results, keyword_scores)],
    key=lambda x: x["ce_score"],
    reverse=True
)

print("BASELINE TOP 3")
for r in keyword_results[:3]:
    print(f"Chunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}")

print("\nRERANKED TOP 3")
for r in keyword_reranked[:3]:
    print(f"Chunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}")

BASELINE TOP 3
Chunk 13 | Pages 5-5
Chunk 34 | Pages 12-12
Chunk 23 | Pages 8-9

RERANKED TOP 3
Chunk 34 | Pages 12-12
Chunk 13 | Pages 5-5
Chunk 142 | Pages 51-52


In [45]:
print("BASELINE TOP 3")
for r in keyword_results[:3]:
    print(f"\nChunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:1200])

print("\n" + "=" * 70)
print("RERANKED TOP 3")
for r in keyword_reranked[:3]:
    print(f"\nChunk {r['chunk_id']} | Pages {r['start_page']}-{r['end_page']}")
    print(r["text"][:1200])

BASELINE TOP 3

Chunk 13 | Pages 5-5
comparison and identification occur only after a significant delay. This involves material, such as pictures or video footage generated by closed circuit television cameras or private devices, which has been generated before the use of the system in respect of the natural persons concerned. (18) The notion of ‘emotion recognition system’ referred to in this Regulation should be defined as an AI system for the purpose of identifying or inferring emotions or intentions of natural persons on the basis of their biometric data. The notion refers to emotions or intentions such as happiness, sadness, anger, surprise, disgust, embarrassment, excitement, shame, contempt, satisfaction and amusement. It does not include physical states, such as pain or fatigue, including, for example, systems used in detecting the state of fatigue of professional pilots or drivers for the purpose of preventing accidents. This does also not include the mere detection of readily

In [46]:
keyword_baseline_prompt = prompt_template.format(
    context=build_context(keyword_results[:3]),
    query=keyword_query
)

keyword_reranked_prompt = prompt_template.format(
    context=build_context(keyword_reranked[:3]),
    query=keyword_query
)

keyword_baseline_answer, keyword_baseline_time = generate_answer(
    keyword_baseline_prompt,
    model_name="gemini-3.5-flash-lite"
)

keyword_reranked_answer, keyword_reranked_time = generate_answer(
    keyword_reranked_prompt,
    model_name="gemini-3.5-flash-lite"
)

print("BASELINE ANSWER\n", keyword_baseline_answer)
print(f"Generation time: {keyword_baseline_time:.2f} seconds")

print("\nRERANKED ANSWER\n", keyword_reranked_answer)
print(f"Generation time: {keyword_reranked_time:.2f} seconds")

BASELINE ANSWER
 Based on the supplied context, AI systems intended to be used to detect the emotional state of individuals in situations related to the workplace are prohibited. 

The exception to this prohibition is that it does not cover AI systems placed on the market strictly for medical or safety reasons, such as systems intended for therapeutical use.
Generation time: 1.90 seconds

RERANKED ANSWER
 Based on the supplied context, the placing on the market, the putting into service for that specific purpose, or the use of AI systems to infer emotions of a natural person in the area of the workplace is prohibited. 

The exception is where the use of the AI system is intended to be put in place or onto the market strictly for medical or safety reasons (such as systems intended for therapeutical use).
Generation time: 1.07 seconds


In [47]:
from pathlib import Path
import pandas as pd

results_dir = Path("/kaggle/working/LMA_RAG_Project/results")

for file in sorted(results_dir.glob("*.csv")):
    df = pd.read_csv(file)
    print(f"\n{file.name} — {df.shape[0]} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
    display(df.head(3))


answer_comparison.csv — 2 rows × 4 columns
Columns: ['pipeline', 'chunks', 'generation_time_seconds', 'answer']


,pipeline,chunks,generation_time_seconds,answer
0,Baseline,"[128, 40, 285]",1.34,"Based on the supplied context, the text mentio..."
1,Cross-Encoder Re-ranked,"[24, 22, 284]",2.21,"Based on the provided context, the prohibited ..."



latency_comparison.csv — 2 rows × 5 columns
Columns: ['pipeline', 'retrieval_s', 'cross_encoder_s', 'generation_s', 'total_s']


,pipeline,retrieval_s,cross_encoder_s,generation_s,total_s
0,Baseline,0.020946,0.000000,1.340780,1.361726
1,Cross-Encoder Re-ranked,0.022090,1.049907,2.212719,3.284716



ranking_comparison.csv — 10 rows × 7 columns
Columns: ['chunk_id', 'original_rank', 'cross_encoder_rank', 'bi_encoder_score', 'cross_encoder_score', 'start_page', 'end_page']


,chunk_id,original_rank,cross_encoder_rank,bi_encoder_score,cross_encoder_score,start_page,end_page
0,24,8,1,0.595963,3.280937,9,9
1,22,4,2,0.614457,1.728470,8,8
2,284,7,3,0.606622,0.207374,121,121



third_answer_comparison.csv — 2 rows × 5 columns
Columns: ['pipeline', 'query', 'chunks', 'generation_time_seconds', 'answer']


,pipeline,query,chunks,generation_time_seconds,answer
0,Baseline,When is biometric categorisation of biometric ...,"[11, 32, 31]",1.91,"Based on the supplied context, the definition ..."
1,Cross-Encoder Re-ranked,When is biometric categorisation of biometric ...,"[11, 24, 32]",0.83,"Based on the provided context, the labelling, ..."



third_ranking_comparison.csv — 10 rows × 6 columns
Columns: ['chunk_id', 'original_rank', 'cross_encoder_rank', 'cross_encoder_score', 'start_page', 'end_page']


,chunk_id,original_rank,cross_encoder_rank,cross_encoder_score,start_page,end_page
0,11,1,1,2.567962,4,5
1,24,8,2,0.428849,9,9
2,32,2,3,-2.311893,11,12


In [49]:
import pandas as pd
from pathlib import Path

results_dir = Path("/kaggle/working/LMA_RAG_Project/results")
results_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([
    {
        "experiment": "Prohibited AI practices",
        "baseline_top3": "[128, 40, 285]",
        "reranked_top3": "[24, 22, 284]",
        "key_rank_shift": "Chunk 24: rank 8 to 1",
        "observation": "Reranked answer was more informative, but incomplete."
    },
    {
        "experiment": "Biometric dataset-labelling exception",
        "baseline_top3": "[11, 32, 31]",
        "reranked_top3": "[11, 24, 32]",
        "key_rank_shift": "Chunk 24: rank 8 to 2",
        "observation": "Reranking retrieved the specific exception missed by baseline."
    },
    {
        "experiment": "Workplace emotion recognition",
        "baseline_top3": "Chunks 13, 34, 23",
        "reranked_top3": "Chunks 34, 13, 142",
        "key_rank_shift": "Chunk 142: rank 9 to 3",
        "observation": "More specific legal provision retrieved; both answers were adequate."
    },
    {
        "experiment": "Biometric categorisation negation",
        "baseline_top3": "Chunks 11, 32, 31",
        "reranked_top3": "Chunks 11, 24, 32",
        "key_rank_shift": "Chunk 24: rank 8 to 2",
        "observation": "Baseline already answered correctly; reranked answer omitted an exception."
    }
])

output_path = results_dir / "evaluation_summary.csv"
summary.to_csv(output_path, index=False)

display(summary)
print(f"\nSaved to: {output_path}")

,experiment,baseline_top3,reranked_top3,key_rank_shift,observation
0,Prohibited AI practices,"[128, 40, 285]","[24, 22, 284]",Chunk 24: rank 8 to 1,"Reranked answer was more informative, but inco..."
1,Biometric dataset-labelling exception,"[11, 32, 31]","[11, 24, 32]",Chunk 24: rank 8 to 2,Reranking retrieved the specific exception mis...
2,Workplace emotion recognition,"Chunks 13, 34, 23","Chunks 34, 13, 142",Chunk 142: rank 9 to 3,More specific legal provision retrieved; both ...
3,Biometric categorisation negation,"Chunks 11, 32, 31","Chunks 11, 24, 32",Chunk 24: rank 8 to 2,Baseline already answered correctly; reranked ...



Saved to: /kaggle/working/LMA_RAG_Project/results/evaluation_summary.csv



## Architecture Comparison: Baseline vs. Cross-Encoder Re-ranking

### 1. Baseline RAG Pipeline

The baseline pipeline uses a Bi-Encoder to retrieve the 10 most relevant chunks from the document based on cosine similarity. It then selects the top 3 chunks directly and passes them to the LLM to generate an answer.

**Pipeline:** Query → Bi-Encoder Retrieval (Top 10) → Select Top 3 → LLM → Answer 1

### 2. Cross-Encoder Re-ranking Pipeline

The re-ranking pipeline first retrieves the same 10 candidate chunks using the Bi-Encoder. A Cross-Encoder then scores each query–chunk pair, and the chunks are sorted by these scores. The top 3 re-ranked chunks are passed to the same LLM using the same prompt structure to generate the answer.

**Pipeline:** Query → Bi-Encoder Retrieval (Top 10) → Cross-Encoder Scoring → Re-ranked Top 3 → LLM → Answer 2

### 3. Architecture Comparison

| Feature | Baseline RAG | Cross-Encoder Re-ranked RAG |
|---|---|---|
| Initial retriever | Bi-Encoder | Bi-Encoder |
| Initial candidate set | Top 10 chunks | Top 10 chunks |
| Ranking method | Cosine similarity between embeddings | Cross-Encoder relevance scores |
| Final context | First 3 retrieved chunks | Top 3 chunks after re-ranking |
| LLM | Same model | Same model |
| Prompt structure | Same prompt template | Same prompt template |
| Main advantage | Faster and computationally efficient | Can improve the relevance of the final context |
| Main limitation | Relevant chunks may rank below the top 3 | Adds computational cost and cannot retrieve chunks excluded from the initial Top 10 |

### 4. Why Is the Cross-Encoder Applied Only to the Top 10?

The document was divided into 315 chunks. The Bi-Encoder efficiently retrieves the top 10 candidates by comparing the query embedding with the stored chunk embeddings. The Cross-Encoder then evaluates only these 10 query–chunk pairs.

Applying the Cross-Encoder to all 315 chunks would require 315 pairwise relevance evaluations per query instead of 10. This would increase computational cost and latency.

The two-stage approach balances efficiency and relevance: the Bi-Encoder performs broad retrieval, while the Cross-Encoder refines the ranking of the candidate chunks. However, re-ranking cannot recover a relevant chunk if the initial Bi-Encoder fails to include it in the Top 10.


In [50]:
import pandas as pd
from pathlib import Path

latency_path = Path(
    "/kaggle/working/LMA_RAG_Project/results/latency_comparison.csv"
)

latency_df = pd.read_csv(latency_path)
display(latency_df.round(4))

,pipeline,retrieval_s,cross_encoder_s,generation_s,total_s
0,Baseline,0.0209,0.0000,1.3408,1.3617
1,Cross-Encoder Re-ranked,0.0221,1.0499,2.2127,3.2847



## Latency Analysis

The latency of the baseline and Cross-Encoder re-ranking pipelines was recorded to understand the computational cost of each approach.

| Stage | Baseline (seconds) | Cross-Encoder Re-ranked (seconds) |
|---|---:|---:|
| Retrieval | 0.0209 | 0.0221 |
| Cross-Encoder scoring and sorting | 0.0000 | 1.0499 |
| LLM generation | 1.3408 | 2.2127 |
| Recorded total | 1.3617 | 3.2847 |

The baseline pipeline has lower recorded latency because it selects the top three chunks directly after Bi-Encoder retrieval. The re-ranked pipeline introduces an additional Cross-Encoder scoring step, which increases computational overhead.

In these recorded runs, the re-ranked pipeline took approximately 1.92 seconds longer overall. However, the LLM generation times were obtained from earlier calls, so the totals should be interpreted as approximate rather than as a controlled end-to-end benchmark. A fair performance comparison would require repeated measurements under the same conditions.



## Answer Quality Analysis

The generated answers from the baseline and Cross-Encoder re-ranking pipelines were compared in terms of relevance, completeness, omissions, and potential unsupported claims.

### 1. Prohibited AI Practices

The baseline answer mainly referred to the relevant articles but stated that the supplied context did not contain enough detail to list the prohibited practices. The re-ranked answer provided more specific examples of prohibited practices.

**Observation:** Re-ranking improved the usefulness of the retrieved context and made the answer more informative. However, the re-ranked answer was still incomplete and did not provide an exhaustive list of prohibited practices.

### 2. Biometric Dataset-Labelling Exception

The baseline answer explained the general definition of biometric categorisation but missed the specific exception concerning lawful labelling, filtering, or categorisation of biometric datasets. The re-ranked answer identified this exception and provided examples such as sorting images by hair or eye colour.

**Observation:** This was a clear improvement in answer relevance because re-ranking promoted a chunk containing the specific exception required by the query.

### 3. Workplace Emotion Recognition

Both pipelines identified the prohibition on workplace emotion recognition and the exception for medical or safety reasons. The re-ranked answer followed the wording of the legal provision more closely, but the baseline answer was already adequate.

**Observation:** Re-ranking improved the specificity of the retrieved context, but the improvement in the final answer was limited.

### 4. Overall Findings

The experiments indicate that Cross-Encoder re-ranking can improve the relevance of the final context by promoting useful chunks that were ranked lower by the Bi-Encoder. However, better rankings do not guarantee better generated answers. The biometric categorisation negation test showed that the baseline answer could be more complete than the re-ranked answer.

No definitive conclusion about hallucination reduction can be drawn from these examples alone. The answers should be checked against the source document to identify unsupported statements, omissions, and incorrect interpretations.



## Failure Case Analysis

### Failure Case 1: Prohibited AI Practices

**Query:** What types of artificial intelligence practices are prohibited under the AI Act?

The baseline pipeline selected chunks 128, 40, and 285. Its answer did not provide specific prohibited practices because the selected context lacked sufficient detail. The Cross-Encoder re-ranked chunk 24 from position 8 to position 1 and chunk 22 from position 4 to position 2. The resulting answer provided more specific examples of prohibited practices.

**Finding:** Re-ranking improved the relevance of the retrieved context and made the answer more informative, although the answer remained incomplete.

### Failure Case 2: Biometric Dataset-Labelling Exception

**Query:** When is biometric categorisation of biometric datasets allowed, such as labelling or filtering images by hair colour or eye colour?

The baseline pipeline selected chunks 11, 32, and 31. It explained biometric categorisation but did not identify the specific dataset-labelling exception. The Cross-Encoder promoted chunk 24 from position 8 to position 2, bringing the relevant exception into the final context.

**Finding:** Re-ranking enabled the answer to identify the exception for lawful labelling, filtering, or categorisation of biometric datasets, including sorting images by hair or eye colour.

### Limitation

These experiments demonstrate improvements in retrieved-context relevance, but they do not establish that re-ranking always improves answer quality. The workplace emotion-recognition experiment showed that the baseline could already answer correctly, while the biometric categorisation negation test showed that a re-ranked answer can omit information present in the baseline answer.


In [51]:
from pathlib import Path

base_dir = Path("/kaggle/working/LMA_RAG_Project")
results_dir = base_dir / "results"
checkpoints_dir = base_dir / "checkpoints"

print("=== RESULTS FILES ===")
for f in sorted(results_dir.iterdir()):
    if f.is_file():
        print(f"✓ {f.name}")

print("\n=== CHECKPOINT FILES ===")
for f in sorted(checkpoints_dir.iterdir()):
    if f.is_file():
        print(f"✓ {f.name}")

=== RESULTS FILES ===
✓ answer_comparison.csv
✓ baseline_answer.json
✓ evaluation_summary.csv
✓ latency_comparison.csv
✓ ranking_comparison.csv
✓ reranked_answer.json
✓ third_answer_comparison.csv
✓ third_ranking_comparison.csv

=== CHECKPOINT FILES ===
✓ chunks.json
✓ embedding_config.json
✓ embeddings.npy
✓ pages.json


In [52]:
import pandas as pd
from pathlib import Path

results_dir = Path("/kaggle/working/LMA_RAG_Project/results")

files_to_check = [
    "ranking_comparison.csv",
    "answer_comparison.csv",
    "third_ranking_comparison.csv",
    "third_answer_comparison.csv",
]

for filename in files_to_check:
    path = results_dir / filename
    print(f"\n{'=' * 60}\n{filename}")

    if path.exists():
        df = pd.read_csv(path)
        display(df)
    else:
        print("File not found.")


ranking_comparison.csv


,chunk_id,original_rank,cross_encoder_rank,bi_encoder_score,cross_encoder_score,start_page,end_page
0,24,8,1,0.595963,3.280937,9,9
1,22,4,2,0.614457,1.728470,8,8
2,284,7,3,0.606622,0.207374,121,121
3,128,1,4,0.640393,-0.654623,44,45
4,285,3,5,0.615521,-1.095464,121,122
5,20,6,6,0.608109,-1.980155,7,8
6,208,10,7,0.591424,-2.606877,82,83
7,120,5,8,0.610291,-4.116801,41,42
8,40,2,9,0.623288,-4.896280,14,14
9,3,9,10,0.592138,-5.127725,2,2



answer_comparison.csv


,pipeline,chunks,generation_time_seconds,answer
0,Baseline,"[128, 40, 285]",1.34,"Based on the supplied context, the text mentio..."
1,Cross-Encoder Re-ranked,"[24, 22, 284]",2.21,"Based on the provided context, the prohibited ..."



third_ranking_comparison.csv


,chunk_id,original_rank,cross_encoder_rank,cross_encoder_score,start_page,end_page
0,11,1,1,2.567962,4,5
1,24,8,2,0.428849,9,9
2,32,2,3,-2.311893,11,12
3,44,7,4,-3.033177,15,16
4,31,3,5,-3.157021,11,11
5,10,4,6,-3.604426,4,5
6,30,6,7,-3.649219,11,11
7,136,10,8,-4.281235,48,49
8,178,9,9,-4.749795,68,69
9,146,5,10,-5.924774,53,53



third_answer_comparison.csv


,pipeline,query,chunks,generation_time_seconds,answer
0,Baseline,When is biometric categorisation of biometric ...,"[11, 32, 31]",1.91,"Based on the supplied context, the definition ..."
1,Cross-Encoder Re-ranked,When is biometric categorisation of biometric ...,"[11, 24, 32]",0.83,"Based on the provided context, the labelling, ..."
